# INFRA: independent validation

This guide imports repository modules. Configure paths explicitly. Synthetic fixtures and trained-model measurements are distinct. Remote sync requires this rebuild to be committed and published first.


In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile

# False uses an existing local checkout. True always creates a NEW disposable
# checkout from Git, avoiding deletion of local work or persistent results.
CLEAN_SYNC = False
PROJECT_GIT_URL = "https://github.com/saltypal/SemanticSchedulerEnd2End.git"
PROJECT_REF = "main"  # Set an exact published commit for reproducible remote runs.
LOCAL_PROJECT = None  # Optional absolute repository path.
RESULTS_DIR = os.environ.get("SEMANTIC_RESULTS_DIR")

if CLEAN_SYNC:
    PROJECT = Path(tempfile.mkdtemp(prefix="semantic-scheduler-")) / "source"
    subprocess.run(["git", "clone", "--no-checkout", PROJECT_GIT_URL, str(PROJECT)], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", PROJECT_REF], check=True)
else:
    candidates = [Path(LOCAL_PROJECT)] if LOCAL_PROJECT else [Path.cwd(), *Path.cwd().parents]
    PROJECT = next((path for path in candidates if (path / "PROJECT_MAP.md").exists()), None)
    if PROJECT is None:
        raise RuntimeError("Set LOCAL_PROJECT or enable CLEAN_SYNC after publishing the rebuild commit")
PROJECT = PROJECT.resolve()
if RESULTS_DIR is None:
    if Path("/kaggle/working").exists():
        RESULTS_DIR = "/kaggle/working/semantic-results"  # Save notebook version to retain Kaggle output.
    elif "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_DIR = "/content/drive/MyDrive/semantic-results"
    else:
        RESULTS_DIR = str(PROJECT.parent / "semantic-results")
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print({"project":str(PROJECT), "ref_requested":PROJECT_REF if CLEAN_SYNC else "local working tree", "results":RESULTS_DIR})


In [ ]:
# The environment owns its torch/CUDA wheel. A checkpoint may require a pinned
# runtime; use EncDec/environment/requirements-model-inference.txt when needed.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,"-m","pip","install","-e", ".[encdec,semparam,scheduler,metrics,notebook]"],check=True)
from INFRA.notebooks import hardware, require_external_results
from INFRA.runs import provenance
RESULTS = require_external_results(PROJECT, RESULTS_DIR)
print(hardware())
print(provenance())


In [ ]:
from INFRA.cli import main
main(["--module","INFRA","--action","demo","--output",str(RESULTS)])
